# Od słów do attention: budujemy blok transformera

## Zadanie
Zbuduj w PyTorchu elementy modelu języka: tokenizację, embeddingi, kodowanie pozycji, skalowane attention, maskę przyczynową, multi-head attention i blok transformera. Sprawdzisz kształty tensorów oraz wpływ maski. To pierwsza część ścieżki; recenzje win są przykładem tekstu.

Wymagane: `torch`, `matplotlib`. Przykłady działają na CPU lub CUDA.

## Rama teoretyczna: transformacja sekwencji przez uwagę

### Reprezentacja problemu
Tekst traktujemy jako uporządkowaną sekwencję dyskretnych zmiennych losowych $x_1,\ldots,x_T$, gdzie każdy token należy do słownika $\mathcal V$ o rozmiarze $V$. Model nie wykonuje obliczeń bezpośrednio na symbolach: tokenizer przypisuje tokenowi indeks $i\in\{0,\ldots,V-1\}$, a macierz embeddingów $E\in\mathbb R^{V\times d}$ mapuje indeks na wektor $e_i=E[i]$. Dla batcha otrzymujemy tensor $X\in\mathbb R^{B\times T\times d}$. Indeks tokenu jest kategorią, a nie miarą odległości: różnica indeksów 17 i 18 nie oznacza podobieństwa semantycznego.

### Od wejścia do zapytań, kluczy i wartości
Self-attention tworzy trzy uczone widoki tej samej reprezentacji: $Q=XW_Q$, $K=XW_K$, $V=XW_V$, gdzie $W_Q,W_K\in\mathbb R^{d\times d_k}$, $W_V\in\mathbb R^{d\times d_v}$. Dla tokenu $i$ skalar $q_i^\top k_j$ mierzy, jak bardzo informacja z pozycji $j$ jest przydatna pozycji $i$. Normalizacja softmax po indeksie $j$ daje rozkład wag $a_{ij}=\exp(s_{ij})/\sum_l\exp(s_{il})$. Wyjście pozycji to $z_i=\sum_j a_{ij}v_j$: każda pozycja buduje kontekstową reprezentację jako ważoną sumę informacji z innych pozycji.

Iloczyn skalarny może rosnąć wraz z wymiarem. Przy niezależnych składowych Q i K o średniej 0 i wariancji 1 wariancja sumy $q^\top k$ jest proporcjonalna do $d_k$. Dzielenie przez $\sqrt{d_k}$ przywraca logity do skali, przy której softmax nie nasyca się nadmiernie i gradienty nie zanikają. Nie jest to arbitralny współczynnik: wynika z wariancji sumy składników.

### Maska, kolejność i przyczynowość
Bez ograniczeń attention każda pozycja może uwzględnić wszystkie pozycje sekwencji. W modelu autoregresyjnym wymagamy $p(x_t\mid x_{<t})$, więc definiujemy $M_{tj}=0$ dla $j\le t$ oraz $M_{tj}=-\infty$ dla $j>t$. W praktyce używamy dużej ujemnej liczby w dtype tensora. Wtedy softmax daje zerową wagę na przyszłość. Maskowanie jest warunkiem poprawnego celu predykcji, a nie tylko wyborem implementacyjnym.

Self-attention bez kodowania pozycji jest permutacyjnie równoważne: permutacja wejściowych embeddingów permutuje wyjście w ten sam sposób, ponieważ wszystkie pozycje są przetwarzane przez te same funkcje. Sam mechanizm podobieństwa nie zna różnicy między „pies goni kota” i „kot goni psa”. Dodanie embeddingu pozycji $P$ do wejścia, $H^{(0)}=E[x]+P$, dostarcza informacji o kolejności. Kod sinusoidalny używa fal o różnych częstotliwościach: $PE_{pos,2i}=\sin(pos/10000^{2i/d})$ i $PE_{pos,2i+1}=\cos(pos/10000^{2i/d})$. Uczone pozycje i względne kodowania (np. RoPE) stanowią alternatywy, ale nie zmieniają podstawowej definicji attention.

### Wiele głów i warstwa transformera
Przy $h$ głowach model dzieli kanały na podprzestrzenie i liczy równolegle $head_r=Attention(XW^Q_r,XW^K_r,XW^V_r)$. Konkatenacja i projekcja $W^O$ pozwalają scalać te reprezentacje. MLP działa niezależnie dla każdej pozycji, ale współdziała z attention: attention miesza informację wzdłuż czasu, MLP przekształca cechy kanałowe. Residual connection $x+f(x)$ zachowuje drogę identycznościową, a LayerNorm kontroluje skalę aktywacji. Typowy pre-LN blok zapisujemy jako $u=x+MHA(LN(x))$, $y=u+MLP(LN(u))$.

Dla sekwencji długości $T$, wymiaru $d$ i $h$ głów macierz wyników podobieństwa ma łącznie $hT^2$ elementów na przykład. Koszt attention jest rzędu $O(T^2d)$, a pamięć wag $O(hT^2)$. Projekcje i MLP mają koszt rzędu $O(Td^2)$. Z tego powodu dla długich sekwencji koszt kwadratowy może dominować, nawet jeśli liczba parametrów pozostaje bez zmian.

### Co wyprowadzi notebook
Kolejne sekcje przełożą powyższe równania na tensory PyTorcha. Zwracaj uwagę na wymiary: batch, głowy, pozycje zapytań/kluczy i kanały. Poprawna implementacja ma zachować kształt $(B,T,d)$ na wyjściu bloku, normalizować wagi po osi kluczy i zerować niedozwolone połączenia causal.

## 1. Korpus i tokenizacja
Pełny Wine Reviews dataset zwykle wymaga osobnego pobrania. Żeby notebook działał offline, korzystamy z małego, dydaktycznego korpusu. Nie jest statystycznie reprezentatywny; można go podmienić na lokalne CSV.

In [ ]:
import math,re,torch
from torch import nn
import matplotlib.pyplot as plt
torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
reviews=["aromatic cherry and plum with a long finish","bright acidity balances ripe citrus fruit","dry wine with notes of pear and mineral","soft tannins frame dark berry and spice","fresh apple aroma with crisp acidity","rich oak vanilla and blackcurrant flavors","light bodied wine with floral notes","ripe peach and honey with a smooth finish","structured tannins and earthy cherry notes","zesty lemon aroma and clean mineral finish","bold dark fruit with pepper and toasted oak","delicate strawberry notes and lively acidity"]
def tokenize(s): return re.findall(r"[a-z]+|[^\w\s]",s.lower())
special=["<pad>","<unk>","<bos>","<eos>"]
vocab=special+sorted({t for s in reviews for t in tokenize(s)})
stoi={t:i for i,t in enumerate(vocab)}
ids=torch.tensor([[stoi.get(t,1) for t in tokenize(reviews[0])]],device=device)
embedding=nn.Embedding(len(vocab),16).to(device)
print("device",device,"vocab",len(vocab),"IDs",ids.shape,"embedding",embedding(ids).shape)

## 2. Scaled dot-product attention
Dla zapytań $Q$, kluczy $K$ i wartości $V$:

$$Attention(Q,K,V)=softmax(QK^T/√d_k+M)V.$$

$QK^T$ mierzy zgodność zapytania z kluczem. Dzielenie przez $√d_k$ stabilizuje wariancję logitów. Maska $M$ dodaje bardzo małą wartość (teoretycznie $-∞$) do zabronionych połączeń.

In [ ]:
def attention(q,k,v,mask=None):
    scores=q@k.transpose(-2,-1)/math.sqrt(q.size(-1))
    if mask is not None:scores=scores.masked_fill(~mask,torch.finfo(scores.dtype).min)
    weights=scores.softmax(-1)
    return weights@v,weights
B,T,dh=2,6,8
q=torch.randn(B,1,T,dh,device=device);k=torch.randn_like(q);v=torch.randn_like(q)
y,a=attention(q,k,v)
print("wyjście",y.shape,"wagi",a.shape,"sumy softmax",a.sum(-1)[0,0])

## 3. Causal masking
GPT przewiduje $x_t$ wyłącznie z $x_{<t}$. Maska dolnotrójkątna zezwala na klucze $j≤t$, blokując przyszłość. `True` oznacza dozwoloną pozycję.

In [ ]:
mask=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
plt.imshow(mask.cpu(),cmap="Greens");plt.xlabel("klucz j");plt.ylabel("zapytanie t");plt.title("Dozwolone połączenia");plt.show()
_,aw=attention(q,k,v,mask)
print("Maksymalna waga z przyszłości:",aw.masked_select(~mask).max().item())

## 4. Multi-head attention
Każda głowa ma własne projekcje i może reprezentować inną relację. Wyniki są konkatenowane i rzutowane: $MHA(X)=Concat(head_i)W^O$. Zwykle $d_k=d_{model}/h$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout=0.):
        super().__init__(); assert d_model%n_heads==0
        self.h=n_heads;self.dh=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model);self.out=nn.Linear(d_model,d_model);self.drop=nn.Dropout(dropout)
    def forward(self,x,mask=None):
        B,T,C=x.shape;q,k,v=self.qkv(x).chunk(3,-1)
        def split(z):return z.view(B,T,self.h,self.dh).transpose(1,2)
        z,w=attention(split(q),split(k),split(v),mask)
        z=z.transpose(1,2).contiguous().view(B,T,C)
        return self.drop(self.out(z)),w
probe=torch.randn(2,7,32,device=device);mha=MultiHeadAttention(32,4).to(device)
y,w=mha(probe,torch.tril(torch.ones(7,7,dtype=torch.bool,device=device)))
print("wejście",probe.shape,"wyjście",y.shape,"mapy",w.shape)

## 5. Kodowanie pozycji
Self-attention bez informacji o kolejności jest równoważne względem permutacji tokenów. Kod sinusoidalny to $PE_{pos,2i}=sin(pos/10000^{2i/d})$ i $PE_{pos,2i+1}=cos(pos/10000^{2i/d})$. GPT często uczy embeddingi pozycji; nowsze warianty używają m.in. RoPE.

In [ ]:
class SinusoidalPosition(nn.Module):
    def __init__(self,d,max_len=256):
        super().__init__();pos=torch.arange(max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.)/d))
        pe=torch.zeros(max_len,d);pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe[None])
    def forward(self,x):return x+self.pe[:,:x.size(1)].to(x.dtype)
pos=SinusoidalPosition(32,64).to(device)
plt.figure(figsize=(8,3));plt.imshow(pos.pe[0].cpu().T,aspect="auto",cmap="coolwarm");plt.xlabel("pozycja");plt.ylabel("wymiar");plt.title("Kodowanie sinusoidalne");plt.show()

## 6. Blok decoder-only
Popularny wariant pre-LayerNorm ma residual attention i MLP: $X'=X+MHA(LN(X))$, $Y=X'+MLP(LN(X'))$. MLP zwykle rozszerza reprezentację do $4d$. GELU wnosi nieliniowość, residual ułatwia przepływ gradientu.

In [ ]:
class Block(nn.Module):
    def __init__(self,d,h,dropout=.1):
        super().__init__();self.ln1=nn.LayerNorm(d);self.attn=MultiHeadAttention(d,h,dropout);self.ln2=nn.LayerNorm(d)
        self.mlp=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d),nn.Dropout(dropout))
    def forward(self,x,mask):
        a,w=self.attn(self.ln1(x),mask);x=x+a;return x+self.mlp(self.ln2(x)),w
block=Block(32,4).to(device);out,maps=block(probe,torch.tril(torch.ones(7,7,dtype=torch.bool,device=device)))
print("wyjście",out.shape,"mapy",maps.shape,"parametry",sum(p.numel() for p in block.parameters()))

## 7. Koszt i ćwiczenia
Dla długości $T$ standardowe attention kosztuje $O(T²d)$ obliczeń i $O(T²)$ pamięci na głowę.

1. Zbuduj maskę lokalną, w której token widzi najwyżej 3 poprzednie pozycje.
2. Zmień liczbę głów przy stałym $d_{model}$ i sprawdź wymiary.
3. Zastąp sinusoidę uczonym `nn.Embedding(max_len,d_model)`.
4. Wyjaśnij, dlaczego permutacja wejścia bez pozycji permutuje wyniki.

## Podsumowanie
Q/K/V wyznaczają ważoną agregację wartości. Skala stabilizuje softmax, maska causal blokuje przyszłość, wiele głów łączy relacje, a kod pozycyjny wnosi kolejność. Blok łączy attention, MLP, LayerNorm i residual.